In [1]:
# ══════════════════════════════════════════════════════════════════════
# Cell 1: Mount Drive
# ══════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════════════
# Cell 1.5: Reassemble labeled chunks into single files per subreddit
# ══════════════════════════════════════════════════════════════════════
import os
import pandas as pd
import gc

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_BASE = os.path.join(BASE_DIR, "labeled_chunks")
OUTPUT_DIR = os.path.join(BASE_DIR, "labeled_complete")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBS = {
    "AITA":      "AITA_labeled.csv",
    "CMV":       "CMV_labeled.csv",
    "T10D":      "T10D_labeled.csv",
    "POLOP":     "POLOP_labeled.csv",
    "UNPOPULAR": "UNPOPULAR_labeled.csv",
}

for sub_prefix, output_filename in SUBS.items():
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    # Skip if already assembled
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        size_mb = os.path.getsize(output_path) / 1e6
        print(f"✅ {output_filename} already exists ({size_mb:.0f} MB) — skipping")
        continue

    chunk_dir = os.path.join(LABELED_BASE, f"{sub_prefix}_labeled")
    if not os.path.isdir(chunk_dir):
        print(f"⚠️  No labeled dir for {sub_prefix}: {chunk_dir} — skipping")
        continue

    chunk_files = sorted([
        f for f in os.listdir(chunk_dir)
        if f.endswith("_labeled.csv") and os.path.getsize(os.path.join(chunk_dir, f)) > 0
    ])

    if not chunk_files:
        print(f"⚠️  No labeled chunks found for {sub_prefix} — skipping")
        continue

    print(f"\n📦 Assembling {sub_prefix} from {len(chunk_files)} chunks...")
    dfs = []
    for cf in chunk_files:
        df = pd.read_csv(os.path.join(chunk_dir, cf), low_memory=False)
        print(f"   {cf}: {len(df):,} rows")
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    print(f"   Combined: {len(combined):,} rows")

    combined.to_csv(output_path, index=False)
    size_mb = os.path.getsize(output_path) / 1e6
    print(f"   ✅ Saved: {output_path} ({size_mb:.0f} MB)")

    del dfs, combined
    gc.collect()

print("\n✅ All done. Engagement pipeline can now load from labeled_complete/")


📦 Assembling AITA from 12 chunks...
   AITA_chunk_000_labeled.csv: 48,330 rows
   AITA_chunk_001_labeled.csv: 46,592 rows
   AITA_chunk_002_labeled.csv: 49,628 rows
   AITA_chunk_003_labeled.csv: 49,513 rows
   AITA_chunk_004_labeled.csv: 49,690 rows
   AITA_chunk_005_labeled.csv: 49,855 rows
   AITA_chunk_006_labeled.csv: 49,975 rows
   AITA_chunk_007_labeled.csv: 49,660 rows
   AITA_chunk_008_labeled.csv: 49,369 rows
   AITA_chunk_009_labeled.csv: 49,632 rows
   AITA_chunk_010_labeled.csv: 49,553 rows
   AITA_chunk_011_labeled.csv: 44,860 rows
   Combined: 586,657 rows
   ✅ Saved: /content/drive/MyDrive/My_Dissent_project/labeled_complete/AITA_labeled.csv (1065 MB)

📦 Assembling CMV from 12 chunks...
   CMV_chunk_000_labeled.csv: 49,603 rows
   CMV_chunk_001_labeled.csv: 49,877 rows
   CMV_chunk_002_labeled.csv: 48,319 rows
   CMV_chunk_003_labeled.csv: 49,925 rows
   CMV_chunk_004_labeled.csv: 49,991 rows
   CMV_chunk_005_labeled.csv: 49,932 rows
   CMV_chunk_006_labeled.csv: 49,89

In [3]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Load all labeled datasets
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import gc
from scipy import stats

BASE_DIR = '/content/drive/MyDrive/My_Dissent_project'
DATA_DIR = os.path.join(BASE_DIR, 'labeled_complete')

FILE_MAP = {
    'AITA_labeled.csv':      'amitheasshole',
    'CMV_labeled.csv':       'changemyview',
    'T10D_labeled.csv':      'the10thdentist',
    'POLOP_labeled.csv':     'politicalopinions',
    'UNPOPULAR_labeled.csv': 'unpopularopinion',
}

frames = []
for filename, sub_name in FILE_MAP.items():
    fpath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(fpath):
        print(f"⚠️  Skipping {filename} — file not found")
        continue
    df = pd.read_csv(fpath, low_memory=False)
    df['subreddit_name'] = sub_name
    df['_source_file'] = filename
    print(f"✅ {filename}: {len(df):,} rows")
    frames.append(df)

raw_df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()

print(f"\n📊 Combined: {len(raw_df):,} rows")
print(f"   Columns: {list(raw_df.columns)}")

✅ AITA_labeled.csv: 586,657 rows
✅ CMV_labeled.csv: 577,825 rows
✅ T10D_labeled.csv: 594,278 rows
✅ POLOP_labeled.csv: 86,493 rows
✅ UNPOPULAR_labeled.csv: 580,579 rows

📊 Combined: 2,425,832 rows
   Columns: ['comment_id', 'comment_body', 'comment_author', 'comment_score', 'comment_created_utc', 'link_id', 'parent_id', 'comment_author_flair_text', 'comment_edited', 'comment_controversiality', 'post_id', 'post_author', 'post_created_utc', 'num_comments', 'post_edited', 'post_score', 'post_title', 'parent_body', 'label', 'label_name', 'confidence', 'subreddit_name', '_source_file']


In [4]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Normalize columns & separate posts / comments
# ══════════════════════════════════════════════════════════════════════
# Every row is a COMMENT with post metadata embedded.
# We extract unique posts and build a proper comments_df.

# ── Comments ──
comments_df = raw_df.copy()
comments_df.rename(columns={
    'comment_id':                'id',
    'comment_body':              'body',
    'comment_author':            'author',
    'comment_score':             'score',
    'comment_created_utc':       'created_utc',
    'comment_author_flair_text': 'author_flair_text',
    'comment_edited':            'edited',
    'comment_controversiality':  'controversiality',
}, inplace=True)

comments_df['id']         = comments_df['id'].astype(str)
comments_df['created_utc'] = pd.to_numeric(comments_df['created_utc'], errors='coerce')
comments_df['link_id']    = comments_df['link_id'].fillna('').astype(str)
comments_df['parent_id']  = comments_df['parent_id'].fillna('').astype(str)
comments_df['post_id']    = comments_df['post_id'].astype(str)
comments_df['post_author'] = comments_df['post_author'].fillna('').astype(str)

# ── Dissent flags ──
# label 0 = substantive_dissent, label 3 = social_disagreement
comments_df['is_dissent'] = (comments_df['label'] == 0)
comments_df['is_social_dissent'] = (comments_df['label'] == 3)
comments_df['is_any_dissent'] = comments_df['label'].isin([0, 3])

# ── Posts (extract unique post metadata from comment rows) ──
post_cols = ['post_id', 'post_author', 'post_title', 'post_score',
             'post_created_utc', 'post_edited', 'num_comments', 'subreddit_name']

posts_df = (
    raw_df[post_cols]
    .dropna(subset=['post_created_utc'])
    .drop_duplicates(subset='post_id', keep='first')
    .rename(columns={
        'post_id':          'id',
        'post_author':      'author',
        'post_title':       'title',
        'post_score':       'score',
        'post_created_utc': 'created_utc',
        'post_edited':      'edited',
    })
    .reset_index(drop=True)
)
posts_df['id']          = posts_df['id'].astype(str)
posts_df['created_utc'] = pd.to_numeric(posts_df['created_utc'], errors='coerce')

print(f"📌 Comments: {len(comments_df):,}")
print(f"📌 Posts:    {len(posts_df):,}")
print(f"\n   Label distribution (all comments):")
print(comments_df['label_name'].value_counts())
print(f"\n   Substantive dissent:  {comments_df['is_dissent'].sum():,} "
      f"({comments_df['is_dissent'].mean()*100:.1f}%)")
print(f"   Social disagreement:  {comments_df['is_social_dissent'].sum():,} "
      f"({comments_df['is_social_dissent'].mean()*100:.1f}%)")
print(f"   Any dissent (sub+soc): {comments_df['is_any_dissent'].sum():,} "
      f"({comments_df['is_any_dissent'].mean()*100:.1f}%)")

# Identify OP vs bystander for H3
UNUSABLE_AUTHORS = {'[deleted]', '[removed]', 'deleted', 'removed', '', 'nan', 'None'}
comments_df['is_op'] = (
    (comments_df['author'] == comments_df['post_author']) &
    (~comments_df['author'].isin(UNUSABLE_AUTHORS)) &
    (comments_df['author'].notna())
)
comments_df['op_identifiable'] = (
    (~comments_df['post_author'].isin(UNUSABLE_AUTHORS)) &
    (comments_df['post_author'].notna())
)
print(f"   OP comments: {comments_df['is_op'].sum():,}")
print(f"   Bystander comments: {(~comments_df['is_op']).sum():,}")
print(f"   Threads with identifiable OP: "
      f"{comments_df.loc[comments_df['op_identifiable'], 'post_id'].nunique():,}")

del raw_df
gc.collect()

📌 Comments: 2,425,832
📌 Posts:    35,569

   Label distribution (all comments):
label_name
substantive_dissent    1489669
agreement               476808
neutral                 347627
social_disagreement     111728
Name: count, dtype: int64

   Substantive dissent:  1,489,669 (61.4%)
   Social disagreement:  111,728 (4.6%)
   Any dissent (sub+soc): 1,601,397 (66.0%)
   OP comments: 239,253
   Bystander comments: 2,186,579
   Threads with identifiable OP: 35,569


7

# 1. Compute Max Reply Depth Per Thread

In [5]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Compute reply depth for all comments
# ══════════════════════════════════════════════════════════════════════

def compute_all_depths(comments_df):
    """
    Iterative depth computation.
    Top-level comments (parent_id starts with 't3_') → depth 1.
    """
    cid_to_parent = dict(zip(comments_df['id'], comments_df['parent_id']))

    depths = {}
    for cid, pid in cid_to_parent.items():
        if pid.startswith('t3_'):
            depths[cid] = 1

    changed = True
    iteration = 0
    while changed:
        changed = False
        iteration += 1
        for cid, pid in cid_to_parent.items():
            if cid in depths:
                continue
            if pid.startswith('t1_'):
                parent_cid = pid[3:]
                if parent_cid in depths:
                    depths[cid] = depths[parent_cid] + 1
                    changed = True
        if iteration % 25 == 0:
            print(f"    ... iteration {iteration}, resolved {len(depths):,} / {len(cid_to_parent):,}")
        if iteration > 200:
            break

    unresolved = 0
    for cid in cid_to_parent:
        if cid not in depths:
            depths[cid] = 1
            unresolved += 1
    if unresolved:
        print(f"    ⚠️ {unresolved:,} comments with unresolved parents defaulted to depth 1")

    return pd.Series(depths, name='depth')


print("Computing comment depths (~2.4M comments, may take a few minutes)...")
depth_series = compute_all_depths(comments_df)
comments_df['depth'] = comments_df['id'].map(depth_series).fillna(1).astype(int)
print(f"  ✅ Max depth found: {comments_df['depth'].max()}")
print(f"  ✅ Mean depth: {comments_df['depth'].mean():.2f}")

Computing comment depths (~2.4M comments, may take a few minutes)...
    ⚠️ 51,920 comments with unresolved parents defaulted to depth 1
  ✅ Max depth found: 240
  ✅ Mean depth: 3.32


# 2. First Dissent Identification & Thread-Level Dissent Variables

In [6]:
# ══════════════════════════════════════════════════════════════════════
# Cell 5: First dissent per thread + dissent intensity variables
# ══════════════════════════════════════════════════════════════════════
# These are the TREATMENT VARIABLES for all three hypotheses.
#   - H1 needs dissent intensity to correlate with engagement & uptake
#   - H2 needs time_to_first_dissent as a moderator
#   - H3 needs first_dissent_utc to define "after dissent" window

# ── First dissent timestamp ──
dissenting_comments = comments_df[comments_df['is_dissent']]
first_dissent = (
    dissenting_comments
    .groupby('post_id')['created_utc']
    .min()
    .rename('first_dissent_utc')
)

print(f"Threads with ≥1 substantive dissent: "
      f"{len(first_dissent):,} / {posts_df['id'].nunique():,}")

# ── Thread-level dissent intensity ──
# These are continuous treatment variables for regression
thread_label_counts = comments_df.groupby('post_id').agg(
    total_comments_in_thread=('id', 'size'),
    n_substantive_dissent=('is_dissent', 'sum'),
    n_social_dissent=('is_social_dissent', 'sum'),
    n_any_dissent=('is_any_dissent', 'sum'),
    n_agreement=('label', lambda x: (x == 1).sum()),
)

# Dissent shares (proportion of thread that is dissent)
thread_label_counts['dissent_share'] = (
    thread_label_counts['n_substantive_dissent'] /
    thread_label_counts['total_comments_in_thread']
)
thread_label_counts['social_dissent_share'] = (
    thread_label_counts['n_social_dissent'] /
    thread_label_counts['total_comments_in_thread']
)
thread_label_counts['any_dissent_share'] = (
    thread_label_counts['n_any_dissent'] /
    thread_label_counts['total_comments_in_thread']
)
thread_label_counts['agreement_share'] = (
    thread_label_counts['n_agreement'] /
    thread_label_counts['total_comments_in_thread']
)

# Binary: does the thread have ANY substantive dissent?
thread_label_counts['has_dissent'] = (
    thread_label_counts['n_substantive_dissent'] > 0
)

# Dissent-to-agreement ratio (for H1: threads where dissent dominates)
thread_label_counts['dissent_agreement_ratio'] = np.where(
    thread_label_counts['n_agreement'] > 0,
    thread_label_counts['n_substantive_dissent'] / thread_label_counts['n_agreement'],
    thread_label_counts['n_substantive_dissent'].clip(upper=10)  # cap for 0-agreement threads
)

print(f"\nThread-level dissent intensity:")
print(thread_label_counts[[
    'dissent_share', 'social_dissent_share',
    'any_dissent_share', 'dissent_agreement_ratio'
]].describe().round(3))

# ── Merge first dissent time onto all comments ──
comments_df = comments_df.merge(first_dissent, on='post_id', how='left')

# Flag: is this comment after the first dissent?
comments_df['after_first_dissent'] = (
    comments_df['created_utc'] > comments_df['first_dissent_utc']
)

# ── Time to first dissent (for H2) ──
# Anchor = post creation time (not first comment — avoids automod noise)
post_created = posts_df.set_index('id')['created_utc'].rename('post_created_utc')
thread_start = comments_df.groupby('post_id')['created_utc'].min().rename('thread_start_utc')
comments_df = comments_df.merge(thread_start, on='post_id', how='left')

# Time from thread start to first dissent (hours)
time_to_first_dissent = (
    (first_dissent - thread_start) / 3600.0
).rename('time_to_first_dissent_hrs')

# Also compute first dissent depth (did dissent come as a top-level reply or deep?)
first_dissent_idx = dissenting_comments.groupby('post_id')['created_utc'].idxmin()
first_dissent_depth = (
    comments_df.loc[first_dissent_idx].set_index('post_id')['depth']
    .rename('first_dissent_depth')
)

print(f"\nTime to first dissent (hours):")
print(time_to_first_dissent.describe().round(2))
print(f"\nFirst dissent depth:")
print(first_dissent_depth.describe().round(2))

Threads with ≥1 substantive dissent: 34,978 / 35,569

Thread-level dissent intensity:
       dissent_share  social_dissent_share  any_dissent_share  \
count      35569.000             35569.000          35569.000   
mean           0.567                 0.041              0.608   
std            0.224                 0.067              0.217   
min            0.000                 0.000              0.000   
25%            0.410                 0.000              0.462   
50%            0.593                 0.000              0.636   
75%            0.740                 0.061              0.778   
max            1.000                 0.909              1.000   

       dissent_agreement_ratio  
count                35569.000  
mean                     5.152  
std                      6.575  
min                      0.000  
25%                      1.333  
50%                      3.000  
75%                      6.667  
max                    144.000  

Time to first dissent (hours):

# 3. Compute All Engagement Metrics Per Thread

In [7]:
# ══════════════════════════════════════════════════════════════════════
# Cell 6: Compute engagement metrics (full thread + after-dissent)
# ══════════════════════════════════════════════════════════════════════
# Per the proposal, Track A includes BOTH full-thread and post-dissent metrics.

def compute_engagement_metrics(comments_df):
    """
    Compute engagement metrics for:
      (a) Full thread (all comments)
      (b) After first dissent only (comments after first_dissent_utc)
    """

    # ── (a) Full-thread metrics ──
    grouped_all = comments_df.groupby('post_id')

    comment_count     = grouped_all.size().rename('comment_count')
    unique_commenters = grouped_all['author'].nunique().rename('unique_commenters')
    max_depth         = grouped_all['depth'].max().rename('max_depth')

    first_time = grouped_all['created_utc'].min()
    last_time  = grouped_all['created_utc'].max()
    thread_lifetime_hrs = ((last_time - first_time) / 3600.0).rename('thread_lifetime_hrs')
    thread_lifetime_hrs = thread_lifetime_hrs.clip(lower=0.0833)

    comment_velocity = (comment_count / thread_lifetime_hrs).rename('comment_velocity')

    # ── (b) After-first-dissent metrics ──
    post_dissent = comments_df[comments_df['after_first_dissent']].copy()
    grouped_pd = post_dissent.groupby('post_id')

    comments_after_dissent           = grouped_pd.size().rename('comments_after_dissent')
    unique_commenters_after_dissent  = grouped_pd['author'].nunique().rename('unique_commenters_after_dissent')
    max_depth_after_dissent          = grouped_pd['depth'].max().rename('max_depth_after_dissent')

    first_pd = grouped_pd['created_utc'].min()
    last_pd  = grouped_pd['created_utc'].max()
    lifetime_after_dissent = ((last_pd - first_pd) / 3600.0).rename('lifetime_after_dissent_hrs')
    lifetime_after_dissent = lifetime_after_dissent.clip(lower=0.0833)

    velocity_after_dissent = (
        comments_after_dissent / lifetime_after_dissent
    ).rename('velocity_after_dissent')

    # ── Combine ──
    engagement = pd.concat([
        # Full thread
        comment_count, unique_commenters, max_depth,
        thread_lifetime_hrs, comment_velocity,
        # After dissent
        comments_after_dissent, unique_commenters_after_dissent,
        max_depth_after_dissent, lifetime_after_dissent, velocity_after_dissent,
    ], axis=1)

    return engagement

print("Computing engagement metrics (full thread + after-dissent)...")
engagement = compute_engagement_metrics(comments_df)
print(f"  Threads with metrics: {len(engagement):,}")
print(f"\n  Full-thread summary:")
print(engagement[['comment_count', 'unique_commenters', 'max_depth',
                   'thread_lifetime_hrs', 'comment_velocity']].describe().round(2))
print(f"\n  After-dissent summary:")
print(engagement[['comments_after_dissent', 'unique_commenters_after_dissent',
                   'max_depth_after_dissent', 'lifetime_after_dissent_hrs',
                   'velocity_after_dissent']].describe().round(2))

Computing engagement metrics (full thread + after-dissent)...
  Threads with metrics: 35,569

  Full-thread summary:
       comment_count  unique_commenters  max_depth  thread_lifetime_hrs  \
count       35569.00           35569.00   35569.00             35569.00   
mean           68.20              40.39       6.65              1242.09   
std           218.19             150.12       6.31              5052.30   
min             1.00               1.00       1.00                 0.08   
25%            11.00               7.00       3.00                 8.59   
50%            21.00              13.00       5.00                30.15   
75%            51.00              27.00       8.00               166.25   
max         15312.00           13354.00     240.00             74290.25   

       comment_velocity  
count          35569.00  
mean               4.41  
std               15.58  
min                0.00  
25%                0.17  
50%                0.80  
75%                2.61  

# 4. Engagement Composite (Z-Score Average)

In [8]:
# ══════════════════════════════════════════════════════════════════════
# Cell 7: Engagement composite WITHOUT max_depth
# ══════════════════════════════════════════════════════════════════════
# max_depth excluded: parent-child chains are incomplete for AITA, CMV,
# and UNPOPULAR due to chunked labeling and source data structure.
# The remaining 4 metrics are unaffected (they use timestamps & authors).

ENGAGEMENT_FULL_COLS = [
    'comment_count', 'unique_commenters',
    'thread_lifetime_hrs', 'comment_velocity',
]

ENGAGEMENT_POST_DISSENT_COLS = [
    'comments_after_dissent', 'unique_commenters_after_dissent',
    'lifetime_after_dissent_hrs', 'velocity_after_dissent',
]

def compute_zscore_composite(df, cols, composite_name):
    """Z-score each column, average into a composite."""
    z_scores = pd.DataFrame(index=df.index)
    for col in cols:
        mean = df[col].mean()
        std  = df[col].std()
        if std == 0 or pd.isna(std):
            z_scores[f'{col}_z'] = 0.0
        else:
            z_scores[f'{col}_z'] = (df[col] - mean) / std

    z_cols = [f'{c}_z' for c in cols]
    df[composite_name] = z_scores[z_cols].mean(axis=1)
    for zc in z_cols:
        df[zc] = z_scores[zc]
    return df

# Full-thread composite
engagement = compute_zscore_composite(
    engagement, ENGAGEMENT_FULL_COLS, 'engagement_composite_full'
)

# After-dissent composite (for H1)
engagement = compute_zscore_composite(
    engagement.copy(), ENGAGEMENT_POST_DISSENT_COLS, 'engagement_composite_postdissent'
)

print("Engagement Composite (full thread) — 4 metrics, no max_depth:")
print(engagement['engagement_composite_full'].describe().round(3))
print(f"\nEngagement Composite (after dissent):")
print(engagement['engagement_composite_postdissent'].dropna().describe().round(3))

Engagement Composite (full thread) — 4 metrics, no max_depth:
count    35569.000
mean         0.000
std          0.640
min         -0.269
25%         -0.227
50%         -0.179
75%         -0.047
max         39.774
Name: engagement_composite_full, dtype: float64

Engagement Composite (after dissent):
count    34803.000
mean         0.000
std          0.640
min         -0.264
25%         -0.228
50%         -0.179
75%         -0.048
max         39.378
Name: engagement_composite_postdissent, dtype: float64


# 5. Merge Treatment Variables + OP/Bystander Stats

In [9]:
# ══════════════════════════════════════════════════════════════════════
# Cell 8: Merge dissent treatment variables + time bins + OP/bystander
# ══════════════════════════════════════════════════════════════════════
# IDEMPOTENT: drops columns before re-merging.

# ── Guard: drop columns if they already exist ──
_cols_to_merge = [
    'time_to_first_dissent_hrs', 'first_dissent_depth',
    'dissent_time_bin',
    'dissent_share', 'social_dissent_share', 'any_dissent_share',
    'agreement_share', 'dissent_agreement_ratio',
    'has_dissent', 'n_substantive_dissent', 'n_social_dissent',
    'total_comments_in_thread', 'n_any_dissent', 'n_agreement',
    'op_replies_after_dissent', 'bystander_replies_after_dissent',
    'op_participated_after_dissent',
    'subreddit_name',
]
_existing = [c for c in _cols_to_merge if c in engagement.columns]
if _existing:
    print(f"  ⚠️ Dropping existing columns for clean re-merge: {_existing}")
    engagement = engagement.drop(columns=_existing)

# ── Time to first dissent ──
engagement = engagement.merge(
    time_to_first_dissent, left_index=True, right_index=True, how='left'
)

# ── First dissent depth ──
engagement = engagement.merge(
    first_dissent_depth, left_index=True, right_index=True, how='left'
)

# ── H2 time bins — finer granularity at the low end ──
engagement['dissent_time_bin'] = pd.cut(
    engagement['time_to_first_dissent_hrs'],
    bins=[0, 1/60, 5/60, 0.5, 3, 12, float('inf')],
    labels=['<1min', '1-5min', '5-30min', '30min-3hrs', '3-12hrs', '12+hrs'],
    right=True
)

# ── Thread-level dissent intensity (from Cell 5) ──
engagement = engagement.merge(
    thread_label_counts, left_index=True, right_index=True, how='left'
)

# ── Subreddit name ──
sub_per_thread = comments_df.groupby('post_id')['subreddit_name'].first()
engagement = engagement.merge(
    sub_per_thread, left_index=True, right_index=True, how='left'
)

# ── H3: OP vs bystander reply counts after dissent ──
post_dissent_comments = comments_df[comments_df['after_first_dissent']].copy()

op_after = post_dissent_comments[post_dissent_comments['is_op']]
op_replies_after = op_after.groupby('post_id').size().rename('op_replies_after_dissent')

bystander_after = post_dissent_comments[~post_dissent_comments['is_op']]
bystander_replies_after = bystander_after.groupby('post_id').size().rename('bystander_replies_after_dissent')

op_participated = (op_replies_after > 0).rename('op_participated_after_dissent')

engagement = engagement.merge(op_replies_after, left_index=True, right_index=True, how='left')
engagement = engagement.merge(bystander_replies_after, left_index=True, right_index=True, how='left')
engagement = engagement.merge(op_participated, left_index=True, right_index=True, how='left')

engagement['op_replies_after_dissent']        = engagement['op_replies_after_dissent'].fillna(0).astype(int)
engagement['bystander_replies_after_dissent'] = engagement['bystander_replies_after_dissent'].fillna(0).astype(int)
engagement['op_participated_after_dissent']   = engagement['op_participated_after_dissent'].fillna(False).astype(bool)

print("H2 — Time-to-first-dissent bins:")
print(engagement['dissent_time_bin'].value_counts().sort_index())
print(f"\nTotal binned: {engagement['dissent_time_bin'].notna().sum():,}")
print(f"NaN (no dissent): {engagement['dissent_time_bin'].isna().sum():,}")

print(f"\nDissent intensity summary:")
print(engagement[['dissent_share', 'social_dissent_share',
                   'dissent_agreement_ratio']].describe().round(3))

print(f"\nH3 — OP participation after dissent:")
has_d = engagement['has_dissent'] == True
print(f"  Threads with dissent: {has_d.sum():,}")
print(f"  OP replied after dissent: "
      f"{engagement.loc[has_d, 'op_participated_after_dissent'].sum():,}")
print(f"  Mean OP replies after dissent:        {engagement['op_replies_after_dissent'].mean():.2f}")
print(f"  Mean bystander replies after dissent:  {engagement['bystander_replies_after_dissent'].mean():.2f}")

H2 — Time-to-first-dissent bins:
dissent_time_bin
<1min         1595
1-5min        6331
5-30min       7419
30min-3hrs    3100
3-12hrs       1081
12+hrs         452
Name: count, dtype: int64

Total binned: 19,978
NaN (no dissent): 15,591

Dissent intensity summary:
       dissent_share  social_dissent_share  dissent_agreement_ratio
count      35569.000             35569.000                35569.000
mean           0.567                 0.041                    5.152
std            0.224                 0.067                    6.575
min            0.000                 0.000                    0.000
25%            0.410                 0.000                    1.333
50%            0.593                 0.000                    3.000
75%            0.740                 0.061                    6.667
max            1.000                 0.909                  144.000

H3 — OP participation after dissent:
  Threads with dissent: 34,978
  OP replied after dissent: 26,309
  Mean OP replies a

/tmp/ipykernel_11059/1541382907.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  engagement['op_participated_after_dissent']   = engagement['op_participated_after_dissent'].fillna(False).astype(bool)


In [12]:
# ══════════════════════════════════════════════════════════════════════
# Cell 9: Merge everything onto posts & save
# ══════════════════════════════════════════════════════════════════════

# Drop subreddit_name from engagement before merge — posts_df already has it
engagement_to_merge = engagement.drop(columns=['subreddit_name'], errors='ignore')

posts_engaged = posts_df.merge(
    engagement_to_merge,
    left_on='id',
    right_index=True,
    how='left'
)

# Clean up any _x/_y suffixes from prior runs
for col in posts_engaged.columns:
    if col.endswith('_x'):
        base = col[:-2]
        posts_engaged.rename(columns={col: base}, inplace=True)
    elif col.endswith('_y'):
        posts_engaged.drop(columns=[col], inplace=True, errors='ignore')

# Engagement level split (median split for quadrant analysis in H1)
has_composite = posts_engaged['engagement_composite_full'].notna()
posts_engaged['engagement_level'] = np.nan
posts_engaged.loc[has_composite, 'engagement_level'] = pd.qcut(
    posts_engaged.loc[has_composite, 'engagement_composite_full'].rank(method='first'),
    q=2,
    labels=['low', 'high']
)

# Dissent terciles (for non-linear effects)
has_dissent_mask = posts_engaged['dissent_share'].notna() & (posts_engaged['dissent_share'] > 0)
posts_engaged['dissent_tercile'] = np.nan
if has_dissent_mask.sum() > 30:
    posts_engaged.loc[has_dissent_mask, 'dissent_tercile'] = pd.qcut(
        posts_engaged.loc[has_dissent_mask, 'dissent_share'].rank(method='first'),
        q=3,
        labels=['low_dissent', 'med_dissent', 'high_dissent']
    )

print(f"📊 Final dataset shape: {posts_engaged.shape}")
print(f"\n   Columns: {list(posts_engaged.columns)}")
print(f"\n   Engagement Level Distribution:")
print(posts_engaged['engagement_level'].value_counts())
print(f"\n   Dissent Tercile Distribution:")
print(posts_engaged['dissent_tercile'].value_counts())
print(f"\n   Threads with dissent: "
      f"{(posts_engaged['has_dissent'] == True).sum():,} / {len(posts_engaged):,}")

print(f"\nMean composites by engagement level:")
print(posts_engaged.groupby('engagement_level', observed=True)[
    ['engagement_composite_full', 'engagement_composite_postdissent']
].mean().round(3))

# Verify subreddit_name survived
print(f"\n   Subreddits: {sorted(posts_engaged['subreddit_name'].dropna().unique())}")

# ── Save posts (thread-level) ──
os.makedirs(BASE_DIR, exist_ok=True)

posts_path = os.path.join(BASE_DIR, 'posts_with_engagement.csv')
posts_engaged.to_csv(posts_path, index=False)
print(f"\n✅ Saved posts: {posts_path}")
print(f"   Shape: {posts_engaged.shape}")

# ── Save comments (comment-level with all computed features) ──
comments_path = os.path.join(BASE_DIR, 'comments_with_features.csv')
print(f"\n💾 Saving comment-level data ({len(comments_df):,} rows)...")
comments_df.to_csv(comments_path, index=False)
comments_size_mb = os.path.getsize(comments_path) / 1e6
print(f"✅ Saved comments: {comments_path} ({comments_size_mb:.0f} MB)")

/tmp/ipykernel_11059/7227696.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['high', 'high', 'low', 'low', 'high', ..., 'low', 'low', 'high', 'low', 'high']
Length: 35569
Categories (2, object): ['low' < 'high']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  posts_engaged.loc[has_composite, 'engagement_level'] = pd.qcut(
/tmp/ipykernel_11059/7227696.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['low_dissent', 'low_dissent', 'med_dissent', 'low_dissent', 'med_dissent', ..., 'high_dissent', 'high_dissent', 'high_dissent', 'med_dissent', 'low_dissent']
Length: 34978
Categories (3, object): ['low_dissent' < 'med_dissent' < 'high_dissent']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  posts_engaged.loc[has_dissent_mask, 'dis

📊 Final dataset shape: (35569, 47)

   Columns: ['id', 'author', 'title', 'score', 'created_utc', 'edited', 'num_comments', 'subreddit_name', 'comment_count', 'unique_commenters', 'max_depth', 'thread_lifetime_hrs', 'comment_velocity', 'comments_after_dissent', 'unique_commenters_after_dissent', 'max_depth_after_dissent', 'lifetime_after_dissent_hrs', 'velocity_after_dissent', 'engagement_composite_full', 'comment_count_z', 'unique_commenters_z', 'thread_lifetime_hrs_z', 'comment_velocity_z', 'engagement_composite_postdissent', 'comments_after_dissent_z', 'unique_commenters_after_dissent_z', 'lifetime_after_dissent_hrs_z', 'velocity_after_dissent_z', 'time_to_first_dissent_hrs', 'first_dissent_depth', 'dissent_time_bin', 'total_comments_in_thread', 'n_substantive_dissent', 'n_social_dissent', 'n_any_dissent', 'n_agreement', 'dissent_share', 'social_dissent_share', 'any_dissent_share', 'agreement_share', 'has_dissent', 'dissent_agreement_ratio', 'op_replies_after_dissent', 'bystander_re

In [13]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10: Per-subreddit diagnostics
# ══════════════════════════════════════════════════════════════════════

SUBREDDITS = sorted(posts_engaged['subreddit_name'].dropna().unique())

print("=" * 75)
print("  ENGAGEMENT METRICS BY SUBREDDIT")
print("=" * 75)

for sub in SUBREDDITS:
    sub_data = posts_engaged[posts_engaged['subreddit_name'] == sub]
    if len(sub_data) == 0:
        continue

    n_with_dissent = (sub_data['has_dissent'] == True).sum()

    print(f"\n{'─' * 60}")
    print(f"  r/{sub}  ({len(sub_data):,} threads, {n_with_dissent:,} with dissent)")
    print(f"{'─' * 60}")

    print(f"  {'FULL THREAD':^55}")
    for col in ENGAGEMENT_FULL_COLS + ['engagement_composite_full']:
        s = sub_data[col].dropna()
        if len(s) == 0:
            continue
        print(f"    {col:40s} median={s.median():9.2f}  mean={s.mean():9.2f}  std={s.std():9.2f}")

    print(f"\n  {'AFTER FIRST DISSENT':^55}")
    for col in ENGAGEMENT_POST_DISSENT_COLS + ['engagement_composite_postdissent']:
        s = sub_data[col].dropna()
        if len(s) == 0:
            print(f"    {col:40s} (no data)")
            continue
        print(f"    {col:40s} median={s.median():9.2f}  mean={s.mean():9.2f}  std={s.std():9.2f}")

    print(f"\n  {'DISSENT INTENSITY':^55}")
    for col in ['dissent_share', 'social_dissent_share', 'dissent_agreement_ratio']:
        s = sub_data[col].dropna()
        if len(s) > 0:
            print(f"    {col:40s} median={s.median():9.3f}  mean={s.mean():9.3f}")

    print(f"\n  {'H2/H3 FEATURES':^55}")
    s = sub_data['time_to_first_dissent_hrs'].dropna()
    if len(s) > 0:
        print(f"    {'time_to_first_dissent_hrs':40s} median={s.median():9.2f}  mean={s.mean():9.2f}")
    print(f"    {'OP participated after dissent':40s} "
          f"{sub_data['op_participated_after_dissent'].sum():,} / {n_with_dissent:,} "
          f"({sub_data['op_participated_after_dissent'].mean()*100:.1f}%)")

  ENGAGEMENT METRICS BY SUBREDDIT

────────────────────────────────────────────────────────────
  r/amitheasshole  (5,893 threads, 5,750 with dissent)
────────────────────────────────────────────────────────────
                        FULL THREAD                      
    comment_count                            median=    21.00  mean=    99.55  std=   376.85
    unique_commenters                        median=    15.00  mean=    73.60  std=   294.40
    thread_lifetime_hrs                      median=    14.57  mean=   199.32  std=   652.93
    comment_velocity                         median=     1.90  mean=     6.53  std=    19.13
    engagement_composite_full                median=    -0.16  mean=     0.07  std=     1.03

                    AFTER FIRST DISSENT                  
    comments_after_dissent                   median=    19.00  mean=    99.20  std=   381.90
    unique_commenters_after_dissent          median=    13.00  mean=    73.12  std=   298.38
    lifetime_after_d

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 11: Label distribution sanity check
# ══════════════════════════════════════════════════════════════════════

print("Label distribution by subreddit:\n")
label_xtab = pd.crosstab(
    comments_df['subreddit_name'],
    comments_df['label_name'],
    normalize='index'
).round(3) * 100

label_xtab.columns = [f'{c} (%)' for c in label_xtab.columns]
print(label_xtab.to_string())

print(f"\n\nDissent rate by subreddit:")
dissent_rate = (
    comments_df
    .groupby('subreddit_name')['is_dissent']
    .mean()
    .sort_values(ascending=False) * 100
).round(1)
for sub, rate in dissent_rate.items():
    print(f"  r/{sub:25s} {rate:5.1f}% dissent")

---
# TRACK A HYPOTHESIS TESTS
# H1: Divergence — Dissent × Engagement Correlation
# H2: Temporal Decay Asymmetry (engagement side)
# H3: Bystander Effect (engagement side)

In [14]:
# ══════════════════════════════════════════════════════════════════════
# Cell 12: H1 — Dissent ↔ Engagement Correlation (Track A side)
# ══════════════════════════════════════════════════════════════════════
# H1 claims: dissent is POSITIVELY correlated with engagement but
# NEGATIVELY correlated (or uncorrelated) with uptake.
#
# Track A establishes the engagement side: dissent_share → engagement.
# Track B (future) will show the uptake side diverges.
#
# Analysis:
#   1. Spearman correlation: dissent_share × engagement_composite
#   2. OLS: engagement ~ dissent_share + controls
#   3. By-subreddit breakdown (effect heterogeneity)
# ══════════════════════════════════════════════════════════════════════

from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Analysis sample: threads with at least 1 dissenting comment ──
h1_df = posts_engaged[
    (posts_engaged['has_dissent'] == True) &
    posts_engaged['engagement_composite_full'].notna() &
    posts_engaged['dissent_share'].notna()
].copy()

print("═" * 70)
print("  H1: DISSENT × ENGAGEMENT CORRELATION (Track A)")
print("═" * 70)
print(f"\n  Analysis sample: {len(h1_df):,} threads with ≥1 dissent")

# ── 1. Overall Spearman correlations ──
print(f"\n{'─' * 70}")
print(f"  1. SPEARMAN CORRELATIONS: dissent_share × engagement metrics")
print(f"{'─' * 70}")

engagement_targets = [
    'engagement_composite_full',
    'engagement_composite_postdissent',
    'comment_count',
    'unique_commenters',
    'thread_lifetime_hrs',
    'comment_velocity',
    'comments_after_dissent',
    'velocity_after_dissent',
]

dissent_predictors = ['dissent_share', 'n_substantive_dissent', 'dissent_agreement_ratio']

print(f"\n  {'':40s} {'dissent_share':>15s} {'n_sub_dissent':>15s} {'d/a ratio':>15s}")
print(f"  {'':40s} {'─'*15} {'─'*15} {'─'*15}")

for target in engagement_targets:
    valid = h1_df[[target] + dissent_predictors].dropna()
    if len(valid) < 30:
        continue

    results = []
    for pred in dissent_predictors:
        rho, p = spearmanr(valid[pred], valid[target])
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        results.append(f"{rho:+.3f}{sig}")

    print(f"  {target:40s} {results[0]:>15s} {results[1]:>15s} {results[2]:>15s}")

# ── 2. By subreddit ──
print(f"\n{'─' * 70}")
print(f"  2. BY-SUBREDDIT: Spearman(dissent_share, engagement_composite_full)")
print(f"{'─' * 70}")

for sub in sorted(h1_df['subreddit_name'].dropna().unique()):
    sub_df = h1_df[h1_df['subreddit_name'] == sub]
    valid = sub_df[['dissent_share', 'engagement_composite_full']].dropna()
    if len(valid) < 30:
        print(f"  r/{sub:25s} n={len(valid):,} (too few)")
        continue
    rho, p = spearmanr(valid['dissent_share'], valid['engagement_composite_full'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  r/{sub:25s} ρ={rho:+.3f}  p={p:.2e}  {sig:4s}  n={len(valid):,}")

# ── 3. OLS: engagement ~ dissent_share + log(thread_size) + subreddit FE ──
print(f"\n{'─' * 70}")
print(f"  3. OLS REGRESSION: engagement_composite_full ~ dissent_share + controls")
print(f"{'─' * 70}")

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    h1_reg = h1_df[[
        'engagement_composite_full', 'dissent_share',
        'total_comments_in_thread', 'subreddit_name'
    ]].dropna().copy()

    h1_reg['log_thread_size'] = np.log1p(h1_reg['total_comments_in_thread'])

    model = smf.ols(
        'engagement_composite_full ~ dissent_share + log_thread_size + C(subreddit_name)',
        data=h1_reg
    ).fit(cov_type='HC1')  # robust SEs

    print(f"\n  N = {int(model.nobs):,}")
    print(f"  R² = {model.rsquared:.4f}")
    print(f"  Adj R² = {model.rsquared_adj:.4f}")
    print(f"\n  Key coefficients:")
    for var in ['Intercept', 'dissent_share', 'log_thread_size']:
        if var in model.params.index:
            coef = model.params[var]
            se = model.bse[var]
            p = model.pvalues[var]
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f"    {var:35s} β={coef:+.4f}  SE={se:.4f}  p={p:.2e} {sig}")

    print(f"\n  ℹ️  Subreddit fixed effects included but not shown.")
    print(f"  ℹ️  Robust standard errors (HC1).")

except ImportError:
    print("  ⚠️ statsmodels not available. Install with: pip install statsmodels")
    print("  Skipping OLS regression.")

# ── 4. Engagement composite by dissent tercile ──
print(f"\n{'─' * 70}")
print(f"  4. ENGAGEMENT BY DISSENT TERCILE")
print(f"{'─' * 70}")

valid_tercile = h1_df[h1_df['dissent_tercile'].notna()]
if len(valid_tercile) > 0:
    tercile_means = valid_tercile.groupby('dissent_tercile', observed=True).agg(
        n=('engagement_composite_full', 'size'),
        eng_full_mean=('engagement_composite_full', 'mean'),
        eng_full_median=('engagement_composite_full', 'median'),
        eng_pd_mean=('engagement_composite_postdissent', 'mean'),
        comment_count_mean=('comment_count', 'mean'),
        velocity_mean=('comment_velocity', 'mean'),
    ).round(3)
    print(tercile_means.to_string())

print(f"\n{'═' * 70}")
print(f"  H1 TRACK A SUMMARY")
print(f"{'═' * 70}")
print(f"  If dissent_share is positively correlated with engagement,")
print(f"  Track A of H1 is supported. Track B (uptake, not yet labeled)")
print(f"  must show the OPPOSITE sign for the full divergence claim.")

══════════════════════════════════════════════════════════════════════
  H1: DISSENT × ENGAGEMENT CORRELATION (Track A)
══════════════════════════════════════════════════════════════════════

  Analysis sample: 34,978 threads with ≥1 dissent

──────────────────────────────────────────────────────────────────────
  1. SPEARMAN CORRELATIONS: dissent_share × engagement metrics
──────────────────────────────────────────────────────────────────────

                                             dissent_share   n_sub_dissent       d/a ratio
                                           ─────────────── ─────────────── ───────────────
  engagement_composite_full                         -0.001       +0.638***       +0.086***
  engagement_composite_postdissent               +0.019***       +0.642***       +0.101***
  comment_count                                  +0.173***       +0.930***       +0.249***
  unique_commenters                              +0.022***       +0.803***       +0.090***
  thr

In [15]:
# ══════════════════════════════════════════════════════════════════════
# Cell 13: H2 — Temporal Decay of Engagement Effect (Track A)
# ══════════════════════════════════════════════════════════════════════
# H2 claims: the engagement-boosting effect of dissent DECAYS with
# time-to-first-dissent, but the uptake effect does NOT (or decays
# more slowly).
#
# Track A establishes: engagement ~ dissent × time_to_first_dissent
#   → we expect a NEGATIVE interaction (late dissent → less engagement)
#
# Analysis:
#   1. Descriptive: engagement composite by time bin
#   2. OLS with interaction: engagement ~ dissent_share × log(time) + controls
#   3. Bin-level Spearman: dissent_share ↔ engagement within each time bin
# ══════════════════════════════════════════════════════════════════════

h2_df = posts_engaged[
    (posts_engaged['has_dissent'] == True) &
    posts_engaged['engagement_composite_full'].notna() &
    posts_engaged['time_to_first_dissent_hrs'].notna() &
    (posts_engaged['time_to_first_dissent_hrs'] >= 0)
].copy()

# Log-transform time (right-skewed)
h2_df['log_time_to_dissent'] = np.log1p(h2_df['time_to_first_dissent_hrs'])

print("═" * 70)
print("  H2: TEMPORAL DECAY OF ENGAGEMENT EFFECT (Track A)")
print("═" * 70)
print(f"\n  Analysis sample: {len(h2_df):,} threads with dissent + valid time")

# ── 1. Descriptive: engagement by time bin ──
print(f"\n{'─' * 70}")
print(f"  1. ENGAGEMENT BY TIME-TO-FIRST-DISSENT BIN")
print(f"{'─' * 70}")

time_bin_stats = h2_df.groupby('dissent_time_bin', observed=True).agg(
    n=('engagement_composite_full', 'size'),
    eng_mean=('engagement_composite_full', 'mean'),
    eng_median=('engagement_composite_full', 'median'),
    eng_pd_mean=('engagement_composite_postdissent', 'mean'),
    velocity_mean=('comment_velocity', 'mean'),
    comments_after_mean=('comments_after_dissent', 'mean'),
    dissent_share_mean=('dissent_share', 'mean'),
).round(3)
print(time_bin_stats.to_string())

# ── 2. Spearman within each time bin ──
print(f"\n{'─' * 70}")
print(f"  2. WITHIN-BIN CORRELATIONS: dissent_share ↔ engagement")
print(f"{'─' * 70}")
print(f"  {'Time Bin':15s} {'n':>7s} {'ρ(full)':>10s} {'p':>10s} {'ρ(post-d)':>10s} {'p':>10s}")

for bin_label in h2_df['dissent_time_bin'].cat.categories:
    bin_df = h2_df[h2_df['dissent_time_bin'] == bin_label]
    valid = bin_df[['dissent_share', 'engagement_composite_full',
                     'engagement_composite_postdissent']].dropna()
    if len(valid) < 30:
        print(f"  {str(bin_label):15s} {len(valid):>7,} (too few)")
        continue

    rho1, p1 = spearmanr(valid['dissent_share'], valid['engagement_composite_full'])
    rho2, p2 = spearmanr(valid['dissent_share'], valid['engagement_composite_postdissent'])
    print(f"  {str(bin_label):15s} {len(valid):>7,} {rho1:>+10.3f} {p1:>10.2e} {rho2:>+10.3f} {p2:>10.2e}")

# ── 3. OLS with interaction term ──
print(f"\n{'─' * 70}")
print(f"  3. OLS: engagement ~ dissent_share * log_time + controls")
print(f"{'─' * 70}")

try:
    import statsmodels.formula.api as smf

    h2_reg = h2_df[[
        'engagement_composite_full', 'dissent_share', 'log_time_to_dissent',
        'total_comments_in_thread', 'subreddit_name'
    ]].dropna().copy()
    h2_reg['log_thread_size'] = np.log1p(h2_reg['total_comments_in_thread'])

    # Model with interaction
    model_h2 = smf.ols(
        'engagement_composite_full ~ dissent_share * log_time_to_dissent '
        '+ log_thread_size + C(subreddit_name)',
        data=h2_reg
    ).fit(cov_type='HC1')

    print(f"\n  N = {int(model_h2.nobs):,}")
    print(f"  R² = {model_h2.rsquared:.4f}")
    print(f"\n  Key coefficients:")
    for var in ['dissent_share', 'log_time_to_dissent',
                'dissent_share:log_time_to_dissent', 'log_thread_size']:
        if var in model_h2.params.index:
            coef = model_h2.params[var]
            se = model_h2.bse[var]
            p = model_h2.pvalues[var]
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f"    {var:45s} β={coef:+.4f}  SE={se:.4f}  p={p:.2e} {sig}")

    print(f"\n  Interpretation:")
    interaction_var = 'dissent_share:log_time_to_dissent'
    if interaction_var in model_h2.params.index:
        int_coef = model_h2.params[interaction_var]
        int_p = model_h2.pvalues[interaction_var]
        if int_coef < 0 and int_p < 0.05:
            print(f"    → NEGATIVE interaction ({int_coef:+.4f}, p={int_p:.2e}):")
            print(f"      Late dissent yields LESS engagement boost. H2 Track A SUPPORTED.")
        elif int_p >= 0.05:
            print(f"    → Interaction NOT significant (p={int_p:.2e}).")
            print(f"      No evidence of temporal decay in engagement effect.")
        else:
            print(f"    → POSITIVE interaction ({int_coef:+.4f}, p={int_p:.2e}):")
            print(f"      Late dissent yields MORE engagement — opposite of H2 prediction.")

except ImportError:
    print("  ⚠️ statsmodels not available.")

# ── 4. Post-dissent engagement by time bin ──
# This is the key for H2: does the engagement bump shrink for late dissent?
print(f"\n{'─' * 70}")
print(f"  4. POST-DISSENT ENGAGEMENT DECAY")
print(f"{'─' * 70}")
print(f"  If H2 holds, 'comments_after_dissent' and 'velocity_after_dissent'")
print(f"  should decrease monotonically across time bins:\n")

decay_cols = ['comments_after_dissent', 'velocity_after_dissent',
              'unique_commenters_after_dissent']
decay_stats = h2_df.groupby('dissent_time_bin', observed=True)[decay_cols].mean().round(2)
print(decay_stats.to_string())

print(f"\n{'═' * 70}")
print(f"  H2 TRACK A SUMMARY")
print(f"{'═' * 70}")
print(f"  Track A (engagement): expect DECAY with time-to-first-dissent.")
print(f"  Track B (uptake, future): expect NO DECAY (or slower decay).")
print(f"  The ASYMMETRY between the two tracks is the H2 claim.")

══════════════════════════════════════════════════════════════════════
  H2: TEMPORAL DECAY OF ENGAGEMENT EFFECT (Track A)
══════════════════════════════════════════════════════════════════════

  Analysis sample: 34,978 threads with dissent + valid time

──────────────────────────────────────────────────────────────────────
  1. ENGAGEMENT BY TIME-TO-FIRST-DISSENT BIN
──────────────────────────────────────────────────────────────────────
                     n  eng_mean  eng_median  eng_pd_mean  velocity_mean  comments_after_mean  dissent_share_mean
dissent_time_bin                                                                                                 
<1min             1595     0.074      -0.138        0.059         10.508               73.304               0.557
1-5min            6331     0.122      -0.125        0.110          7.977              104.948               0.547
5-30min           7419     0.008      -0.169        0.002          3.820               65.959        

In [16]:
# ══════════════════════════════════════════════════════════════════════
# Cell 14: H3 — Bystander vs OP Engagement After Dissent (Track A)
# ══════════════════════════════════════════════════════════════════════
# H3 claims: dissent produces more uptake among BYSTANDERS than the
# direct target (OP), moderated by evidence cues.
#
# Track A can only measure ENGAGEMENT (reply rates, comment counts)
# not actual stance change. But it establishes:
#   - Do bystanders engage MORE after dissent than OP?
#   - Is the bystander engagement share higher in high-dissent threads?
#   - Are there more NEW participants (not OP) after dissent?
#
# Evidence cue moderation requires Track B labels. Here we prepare
# the evidence cue proxy: does the dissenting comment contain
# links, quotes, or structured arguments? (simple heuristic for now)
# ══════════════════════════════════════════════════════════════════════

print("═" * 70)
print("  H3: BYSTANDER vs OP ENGAGEMENT AFTER DISSENT (Track A)")
print("═" * 70)

# ── Analysis sample: threads with identifiable OP + dissent ──
h3_threads = posts_engaged[
    (posts_engaged['has_dissent'] == True) &
    posts_engaged['engagement_composite_full'].notna()
].copy()

# Merge OP identifiability
op_identifiable_per_thread = (
    comments_df.groupby('post_id')['op_identifiable'].first()
)
h3_threads = h3_threads.merge(
    op_identifiable_per_thread, left_on='id', right_index=True, how='left'
)
h3_threads = h3_threads[h3_threads['op_identifiable'] == True]

print(f"\n  H3 sample: {len(h3_threads):,} threads (identifiable OP + dissent)")

# ── 1. OP vs Bystander reply counts after dissent ──
print(f"\n{'─' * 70}")
print(f"  1. OP vs BYSTANDER REPLIES AFTER DISSENT")
print(f"{'─' * 70}")

# Bystander share of post-dissent activity
h3_threads['total_replies_after_dissent'] = (
    h3_threads['op_replies_after_dissent'] +
    h3_threads['bystander_replies_after_dissent']
)
h3_threads['bystander_share_after'] = np.where(
    h3_threads['total_replies_after_dissent'] > 0,
    h3_threads['bystander_replies_after_dissent'] / h3_threads['total_replies_after_dissent'],
    np.nan
)

print(f"\n  Overall (all threads with dissent):")
print(f"    Mean OP replies after dissent:        {h3_threads['op_replies_after_dissent'].mean():.2f}")
print(f"    Mean bystander replies after dissent: {h3_threads['bystander_replies_after_dissent'].mean():.2f}")
print(f"    Mean bystander share of post-dissent: {h3_threads['bystander_share_after'].mean():.3f}")
print(f"    OP participated at all after dissent: "
      f"{h3_threads['op_participated_after_dissent'].sum():,} / {len(h3_threads):,} "
      f"({h3_threads['op_participated_after_dissent'].mean()*100:.1f}%)")

# ── 2. By subreddit ──
print(f"\n{'─' * 70}")
print(f"  2. OP vs BYSTANDER BY SUBREDDIT")
print(f"{'─' * 70}")
print(f"  {'Subreddit':25s} {'n':>7s} {'OP_mean':>9s} {'Byst_mean':>10s} {'Byst_%':>8s} {'OP_part%':>9s}")

for sub in sorted(h3_threads['subreddit_name'].dropna().unique()):
    s = h3_threads[h3_threads['subreddit_name'] == sub]
    print(f"  {sub:25s} {len(s):>7,} "
          f"{s['op_replies_after_dissent'].mean():>9.2f} "
          f"{s['bystander_replies_after_dissent'].mean():>10.2f} "
          f"{s['bystander_share_after'].mean()*100:>7.1f}% "
          f"{s['op_participated_after_dissent'].mean()*100:>8.1f}%")

# ── 3. Does bystander engagement increase with dissent intensity? ──
print(f"\n{'─' * 70}")
print(f"  3. BYSTANDER ENGAGEMENT BY DISSENT INTENSITY")
print(f"{'─' * 70}")

valid_tercile_h3 = h3_threads[h3_threads['dissent_tercile'].notna()]
if len(valid_tercile_h3) > 0:
    tercile_h3 = valid_tercile_h3.groupby('dissent_tercile', observed=True).agg(
        n=('id', 'size'),
        op_replies_mean=('op_replies_after_dissent', 'mean'),
        bystander_replies_mean=('bystander_replies_after_dissent', 'mean'),
        bystander_share=('bystander_share_after', 'mean'),
        op_participation_rate=('op_participated_after_dissent', 'mean'),
    ).round(3)
    print(tercile_h3.to_string())

# ── 4. Correlation: dissent_share → bystander share ──
print(f"\n{'─' * 70}")
print(f"  4. CORRELATION: dissent_share → bystander_share_after")
print(f"{'─' * 70}")

valid_byst = h3_threads[['dissent_share', 'bystander_share_after']].dropna()
if len(valid_byst) >= 30:
    rho, p = spearmanr(valid_byst['dissent_share'], valid_byst['bystander_share_after'])
    print(f"  Spearman ρ = {rho:+.3f}, p = {p:.2e}, n = {len(valid_byst):,}")
    if rho > 0 and p < 0.05:
        print(f"  → More dissent → higher bystander share of activity.")

# ── 5. Evidence cue proxy (heuristic: links, quotes, data) ──
print(f"\n{'─' * 70}")
print(f"  5. EVIDENCE CUE PROXY (for future moderation analysis)")
print(f"{'─' * 70}")

# Tag first dissenting comment per thread with evidence cues
first_dissent_comments = comments_df.loc[
    dissenting_comments.groupby('post_id')['created_utc'].idxmin()
].copy()

# Heuristic evidence cues
first_dissent_comments['has_link'] = (
    first_dissent_comments['body'].str.contains(r'https?://', na=False, regex=True)
)
first_dissent_comments['has_quote'] = (
    first_dissent_comments['body'].str.contains(r'^\s*>', na=False, regex=True) |
    first_dissent_comments['body'].str.contains(r'"[^"]{20,}"', na=False, regex=True)
)
first_dissent_comments['has_data'] = (
    first_dissent_comments['body'].str.contains(
        r'\d+%|study|research|according to|data|evidence|source',
        na=False, case=False, regex=True
    )
)
first_dissent_comments['has_evidence_cue'] = (
    first_dissent_comments['has_link'] |
    first_dissent_comments['has_quote'] |
    first_dissent_comments['has_data']
)

evidence_per_thread = (
    first_dissent_comments
    .set_index('post_id')[['has_link', 'has_quote', 'has_data', 'has_evidence_cue']]
)

n_evidence = first_dissent_comments['has_evidence_cue'].sum()
n_total_fd = len(first_dissent_comments)
print(f"  First dissenting comments with evidence cues: "
      f"{n_evidence:,} / {n_total_fd:,} ({n_evidence/n_total_fd*100:.1f}%)")
print(f"    has_link:  {first_dissent_comments['has_link'].sum():,} ({first_dissent_comments['has_link'].mean()*100:.1f}%)")
print(f"    has_quote: {first_dissent_comments['has_quote'].sum():,} ({first_dissent_comments['has_quote'].mean()*100:.1f}%)")
print(f"    has_data:  {first_dissent_comments['has_data'].sum():,} ({first_dissent_comments['has_data'].mean()*100:.1f}%)")

# Save evidence cues for Track B
h3_threads = h3_threads.merge(
    evidence_per_thread, left_on='id', right_index=True, how='left'
)

# Engagement by evidence cue (preview)
for cue_col in ['has_evidence_cue', 'has_link', 'has_data']:
    valid = h3_threads[h3_threads[cue_col].notna()]
    if len(valid) < 30:
        continue
    grp = valid.groupby(cue_col).agg(
        n=('id', 'size'),
        eng=('engagement_composite_full', 'mean'),
        bystander_share=('bystander_share_after', 'mean'),
    ).round(3)
    print(f"\n  {cue_col}:")
    print(f"  {grp.to_string()}")

print(f"\n{'═' * 70}")
print(f"  H3 TRACK A SUMMARY")
print(f"{'═' * 70}")
print(f"  Track A establishes that bystanders dominate post-dissent activity.")
print(f"  Track B (uptake) will test whether bystanders also show more")
print(f"  stance-change than OP, moderated by evidence cues.")
print(f"  Evidence cue flags are saved for that analysis.")

══════════════════════════════════════════════════════════════════════
  H3: BYSTANDER vs OP ENGAGEMENT AFTER DISSENT (Track A)
══════════════════════════════════════════════════════════════════════

  H3 sample: 34,978 threads (identifiable OP + dissent)

──────────────────────────────────────────────────────────────────────
  1. OP vs BYSTANDER REPLIES AFTER DISSENT
──────────────────────────────────────────────────────────────────────

  Overall (all threads with dissent):
    Mean OP replies after dissent:        6.65
    Mean bystander replies after dissent: 60.21
    Mean bystander share of post-dissent: 0.829
    OP participated at all after dissent: 26,309 / 34,978 (75.2%)

──────────────────────────────────────────────────────────────────────
  2. OP vs BYSTANDER BY SUBREDDIT
──────────────────────────────────────────────────────────────────────
  Subreddit                       n   OP_mean  Byst_mean   Byst_%  OP_part%
  amitheasshole               5,750      4.74      94.07 

In [17]:
# ══════════════════════════════════════════════════════════════════════
# Cell 15: Track A Summary Table (for the paper)
# ══════════════════════════════════════════════════════════════════════

print("═" * 75)
print("  TRACK A (ENGAGEMENT) — RESULTS SUMMARY")
print("═" * 75)

print(f"""
┌───────────────────────────────────────────────────────────────────────┐
│ DATASET                                                             │
├───────────────────────────────────────────────────────────────────────┤
│ Total comments labeled:   {len(comments_df):>10,}                              │
│ Total threads:            {len(posts_engaged):>10,}                              │
│ Threads with dissent:     {(posts_engaged['has_dissent']==True).sum():>10,}                              │
│ Subreddits:               {posts_engaged['subreddit_name'].nunique():>10,}                              │
│ Labeling coverage:             98.6%                                │
├───────────────────────────────────────────────────────────────────────┤
│ TRACK A METRICS (4-item Z-score composite, excl. max_depth)        │
│   • Comment count                                                   │
│   • Unique commenters                                               │
│   • Thread lifetime (hrs)                                           │
│   • Comment velocity (comments/hr)                                  │
├───────────────────────────────────────────────────────────────────────┤
│ TREATMENT VARIABLES                                                 │
│   • dissent_share (continuous, 0-1)                                 │
│   • time_to_first_dissent_hrs (continuous)                          │
│   • has_evidence_cue (binary)                                       │
│   • OP vs bystander (comment-level role flag)                       │
└───────────────────────────────────────────────────────────────────────┘
""")

# Quick hypothesis support summary
print("HYPOTHESIS SUPPORT (Track A only):")
print("─" * 50)

# H1 check
h1_valid = posts_engaged[
    (posts_engaged['has_dissent']==True) &
    posts_engaged['dissent_share'].notna() &
    posts_engaged['engagement_composite_full'].notna()
]
if len(h1_valid) >= 30:
    rho_h1, p_h1 = spearmanr(h1_valid['dissent_share'], h1_valid['engagement_composite_full'])
    print(f"  H1: dissent_share ↔ engagement  ρ={rho_h1:+.3f} p={p_h1:.2e}")
    if rho_h1 > 0 and p_h1 < 0.05:
        print(f"       → POSITIVE correlation. Track A SUPPORTED.")
        print(f"       → Awaiting Track B for divergence test.")
    else:
        print(f"       → Not significantly positive. Track A NOT supported.")

print()

# H2 check (descriptive)
h2_early = posts_engaged[
    (posts_engaged['has_dissent']==True) &
    (posts_engaged['time_to_first_dissent_hrs'] <= 0.5)
]['engagement_composite_postdissent'].mean()
h2_late = posts_engaged[
    (posts_engaged['has_dissent']==True) &
    (posts_engaged['time_to_first_dissent_hrs'] > 3)
]['engagement_composite_postdissent'].mean()
print(f"  H2: Post-dissent engagement (early ≤30min): {h2_early:.3f}")
print(f"      Post-dissent engagement (late >3hrs):   {h2_late:.3f}")
if not np.isnan(h2_early) and not np.isnan(h2_late):
    if h2_early > h2_late:
        print(f"       → Engagement decays. Track A SUPPORTED.")
        print(f"       → Awaiting Track B to test decay asymmetry.")
    else:
        print(f"       → No decay pattern. Track A NOT supported.")

print()

# H3 check
h3_valid = posts_engaged[
    (posts_engaged['has_dissent']==True) &
    posts_engaged['bystander_replies_after_dissent'].notna()
]
mean_byst = h3_valid['bystander_replies_after_dissent'].mean()
mean_op = h3_valid['op_replies_after_dissent'].mean()
print(f"  H3: Mean bystander replies after dissent: {mean_byst:.2f}")
print(f"      Mean OP replies after dissent:        {mean_op:.2f}")
if mean_byst > mean_op:
    print(f"       → Bystanders MORE active. Track A SUPPORTED.")
    print(f"       → Awaiting Track B for stance-change analysis.")

print(f"\n{'═' * 75}")
print(f"  FILES SAVED:")
print(f"    posts_with_engagement.csv  — thread-level (all metrics + treatment vars)")
print(f"    comments_with_features.csv — comment-level (labels, depth, timestamps)")
print(f"{'═' * 75}")

═══════════════════════════════════════════════════════════════════════════
  TRACK A (ENGAGEMENT) — RESULTS SUMMARY
═══════════════════════════════════════════════════════════════════════════

┌───────────────────────────────────────────────────────────────────────┐
│ DATASET                                                             │
├───────────────────────────────────────────────────────────────────────┤
│ Total comments labeled:    2,425,832                              │
│ Total threads:                35,569                              │
│ Threads with dissent:         34,978                              │
│ Subreddits:                        5                              │
│ Labeling coverage:             98.6%                                │
├───────────────────────────────────────────────────────────────────────┤
│ TRACK A METRICS (4-item Z-score composite, excl. max_depth)        │
│   • Comment count                                                   │
│   • Unique comm

In [18]:
# ══════════════════════════════════════════════════════════════════════
# Cell 16 (Diagnostic): Thread completeness analysis
# ══════════════════════════════════════════════════════════════════════

all_ids = set(comments_df['id'])

comments_df['_parent_type'] = np.where(
    comments_df['parent_id'].str.startswith('t3_'), 'top_level',
    np.where(comments_df['parent_id'].str.startswith('t1_'), 'reply', 'other')
)

reply_mask = comments_df['_parent_type'] == 'reply'
comments_df['_parent_found'] = False
comments_df.loc[reply_mask, '_parent_found'] = (
    comments_df.loc[reply_mask, 'parent_id'].str[3:].isin(all_ids)
)

thread_stats = comments_df.groupby('post_id').agg(
    total_comments=('id', 'size'),
    top_level=('_parent_type', lambda x: (x == 'top_level').sum()),
    replies=('_parent_type', lambda x: (x == 'reply').sum()),
    replies_with_parent_found=('_parent_found', 'sum'),
).reset_index()

thread_stats['replies_orphaned'] = thread_stats['replies'] - thread_stats['replies_with_parent_found']
thread_stats['orphan_rate'] = np.where(
    thread_stats['replies'] > 0,
    thread_stats['replies_orphaned'] / thread_stats['replies'],
    0.0
)
thread_stats['is_complete'] = (thread_stats['replies_orphaned'] == 0)

print("=" * 70)
print("  THREAD COMPLETENESS ANALYSIS")
print("=" * 70)

n_total = len(thread_stats)
n_complete = thread_stats['is_complete'].sum()

print(f"\n  Total threads:      {n_total:,}")
print(f"  Complete threads:   {n_complete:,}  ({n_complete/n_total*100:.1f}%)")
print(f"  Incomplete threads: {n_total - n_complete:,}  ({(n_total - n_complete)/n_total*100:.1f}%)")

complete_ids = set(thread_stats.loc[thread_stats['is_complete'], 'post_id'])
comments_in_complete = comments_df['post_id'].isin(complete_ids).sum()
print(f"\n  Comments in complete threads:   {comments_in_complete:,} / {len(comments_df):,} "
      f"({comments_in_complete/len(comments_df)*100:.1f}%)")

print(f"\n  Orphan rate distribution:")
print(thread_stats['orphan_rate'].describe().round(3))

sub_map = comments_df.groupby('post_id')['subreddit_name'].first()
thread_stats['subreddit'] = thread_stats['post_id'].map(sub_map)

print(f"\n{'─' * 70}")
print(f"  BY SUBREDDIT")
print(f"{'─' * 70}")
for sub in sorted(thread_stats['subreddit'].dropna().unique()):
    sub_threads = thread_stats[thread_stats['subreddit'] == sub]
    n = len(sub_threads)
    n_comp = sub_threads['is_complete'].sum()
    mean_orphan = sub_threads['orphan_rate'].mean()
    print(f"  r/{sub:25s} {n:,} threads, {n_comp:,} complete ({n_comp/n*100:.1f}%), "
          f"orphan rate {mean_orphan*100:.1f}%")

comments_df.drop(columns=['_parent_type', '_parent_found'], inplace=True)

  THREAD COMPLETENESS ANALYSIS

  Total threads:      35,569
  Complete threads:   31,348  (88.1%)
  Incomplete threads: 4,221  (11.9%)

  Comments in complete threads:   1,501,964 / 2,425,832 (61.9%)

  Orphan rate distribution:
count    35569.000
mean         0.010
std          0.054
min          0.000
25%          0.000
50%          0.000
75%          0.000
max          1.000
Name: orphan_rate, dtype: float64

──────────────────────────────────────────────────────────────────────
  BY SUBREDDIT
──────────────────────────────────────────────────────────────────────
  r/amitheasshole             5,893 threads, 5,141 complete (87.2%), orphan rate 1.1%
  r/changemyview              5,200 threads, 3,979 complete (76.5%), orphan rate 1.3%
  r/politicalopinions         4,427 threads, 4,212 complete (95.1%), orphan rate 0.7%
  r/the10thdentist            7,885 threads, 7,275 complete (92.3%), orphan rate 0.7%
  r/unpopularopinion          12,164 threads, 10,741 complete (88.3%), orphan rate

In [19]:
# ══════════════════════════════════════════════════════════════════════
# Cell 17 (Diagnostic): Orphan analysis — Reddit vs chunking artifact
# ══════════════════════════════════════════════════════════════════════

all_ids = set(comments_df['id'])

reply_mask = comments_df['parent_id'].str.startswith('t1_')
comments_df['_parent_ref'] = np.where(reply_mask, comments_df['parent_id'].str[3:], np.nan)
comments_df['_parent_found'] = comments_df['_parent_ref'].isin(all_ids)
comments_df.loc[~reply_mask, '_parent_found'] = True

print("=" * 70)
print("  ORPHAN ANALYSIS: BASE RATE vs CHUNKING ARTIFACT")
print("=" * 70)

for sub in sorted(comments_df['subreddit_name'].unique()):
    sub_df = comments_df[comments_df['subreddit_name'] == sub]
    n_total = len(sub_df)
    n_replies = reply_mask[sub_df.index].sum()
    n_found = sub_df.loc[reply_mask[sub_df.index], '_parent_found'].sum()
    n_orphaned = n_replies - n_found

    print(f"\n  r/{sub}")
    print(f"    Total: {n_total:,}  Replies: {n_replies:,}  Orphaned: {n_orphaned:,} "
          f"({n_orphaned/max(n_replies,1)*100:.1f}%)")

print(f"\n{'═' * 70}")
print("  If T10D & POLOP (fully labeled) have similar orphan rates to")
print("  AITA/CMV/UNPOPULAR, it's a Reddit data issue, not chunking.")
print(f"{'═' * 70}")

comments_df.drop(columns=['_parent_ref', '_parent_found'], inplace=True, errors='ignore')

  ORPHAN ANALYSIS: BASE RATE vs CHUNKING ARTIFACT

  r/amitheasshole
    Total: 586,657  Replies: 260,552  Orphaned: 4,222 (1.6%)

  r/changemyview
    Total: 577,825  Replies: 465,105  Orphaned: 4,659 (1.0%)

  r/politicalopinions
    Total: 86,493  Replies: 61,921  Orphaned: 316 (0.5%)

  r/the10thdentist
    Total: 594,278  Replies: 340,058  Orphaned: 1,695 (0.5%)

  r/unpopularopinion
    Total: 580,579  Replies: 327,513  Orphaned: 4,846 (1.5%)

══════════════════════════════════════════════════════════════════════
  If T10D & POLOP (fully labeled) have similar orphan rates to
  AITA/CMV/UNPOPULAR, it's a Reddit data issue, not chunking.
══════════════════════════════════════════════════════════════════════


In [20]:
# ══════════════════════════════════════════════════════════════════════
# Cell 18 (Diagnostic): OP identity analysis
# ══════════════════════════════════════════════════════════════════════

print("=" * 70)
print("  OP IDENTITY ANALYSIS")
print("=" * 70)

print("\nMost common post_author values:")
print(comments_df['post_author'].value_counts().head(15))

thread_op_status = comments_df.groupby('post_id').agg(
    post_author=('post_author', 'first'),
    op_usable=('op_identifiable', 'first'),
    subreddit=('subreddit_name', 'first'),
    total_comments=('id', 'size'),
).reset_index()

n_total = len(thread_op_status)
n_usable = thread_op_status['op_usable'].sum()

print(f"\n  Total threads:           {n_total:,}")
print(f"  OP identifiable:         {n_usable:,} ({n_usable/n_total*100:.1f}%)")
print(f"  OP deleted/missing:      {n_total - n_usable:,} ({(n_total-n_usable)/n_total*100:.1f}%)")

print(f"\n{'─' * 70}")
print(f"  BY SUBREDDIT")
print(f"{'─' * 70}")
for sub in sorted(thread_op_status['subreddit'].dropna().unique()):
    sub_df = thread_op_status[thread_op_status['subreddit'] == sub]
    n = len(sub_df)
    n_good = sub_df['op_usable'].sum()
    print(f"  r/{sub:25s} {n_good:,} / {n:,} usable ({n_good/n*100:.1f}%)")

# Impact on H3
has_dissent_set = set(comments_df.loc[comments_df['is_dissent'], 'post_id'])
thread_op_status['has_dissent'] = thread_op_status['post_id'].isin(has_dissent_set)
h3_eligible = thread_op_status[
    thread_op_status['op_usable'] & thread_op_status['has_dissent']
]

print(f"\n  H3 eligible (usable OP + dissent): {len(h3_eligible):,} threads")

op_comments_after = comments_df[
    comments_df['after_first_dissent'] &
    comments_df['is_op'] &
    comments_df['op_identifiable']
]
print(f"  Of those, OP actually replied after dissent: "
      f"{op_comments_after['post_id'].nunique():,} threads")

  OP IDENTITY ANALYSIS

Most common post_author values:
post_author
Rema5000                15312
throwawayscraps          6063
Inside_Register3070      5634
lionprincesslioness      5423
PrimNathanIOW            5370
glowshroom12             5367
Electrical_Box7293       5191
Lost_Roku_Remote         4987
aitarudecelebrity        4919
UnpopularOpinionMods     4641
FriesWithMacSauce        4085
tidylinks                4050
IchumbachiB              4046
UnauthorizedFart         3930
HiBreek                  3924
Name: count, dtype: int64

  Total threads:           35,569
  OP identifiable:         35,569 (100.0%)
  OP deleted/missing:      0 (0.0%)

──────────────────────────────────────────────────────────────────────
  BY SUBREDDIT
──────────────────────────────────────────────────────────────────────
  r/amitheasshole             5,893 / 5,893 usable (100.0%)
  r/changemyview              5,200 / 5,200 usable (100.0%)
  r/politicalopinions         4,427 / 4,427 usable (100.0%)
  r/